In [124]:
from data.constants import DNC_START, DNC_END
import pandas as pd

In [125]:
train_panel_in = "../../../data/final/train_panel.parquet"
bike_panel_in = "../../../data/final/bike_panel.parquet"
uber_panel_in = "../../../data/final/uber_panel.parquet"

bal_out = "../../../reports/replication/balance.html"

In [126]:
train_panel = pd.read_parquet(train_panel_in)
bike_panel = pd.read_parquet(bike_panel_in)
uber_panel = pd.read_parquet(uber_panel_in)

In [127]:
panel = pd.concat([train_panel, bike_panel, uber_panel],
                     keys=['train', 'bike', 'uber'], names=['transit']).reset_index('transit')
assert not any(panel.groupby('id')['transit'].nunique() > 1), 'ids must be unique across transits'

In [128]:
import pandas as pd
import numpy as np

rows = []
for (t,g), sub in panel.groupby(['transit','UCMP']):
    pre = sub[(sub['date'] < DNC_START) | (sub['date'] > DNC_END)]
    post = sub[(sub['date'] >= DNC_START) & (sub['date'] <= DNC_END)]
    rows.append({
        "Group": "Treated" if g == 1 else "Control",
        "Mode": t,
        "N units": sub['id'].nunique(),
        "N unit-days": len(sub),
        "N pre": len(pre),
        "N post": len(post),
        "Mean log(rides) (pre)": pre['logrides'].mean(),
        "Mean log(rides) (post)": post['logrides'].mean(),
        "% complete panel": (sub.groupby('id').size()
                             .eq(panel['date'].nunique()).mean()),
    })
bal = pd.DataFrame(rows).T
bal = bal.map(lambda v: f"{v:.3f}" if isinstance(v, float) else v)
from tabulate import tabulate
header_rows = list(map(lambda x: f'{x[0]}\n{x[1]}', zip(['bike', ' ', 'train', ' ', 'uber', ' '], bal.iloc[0])))
bal_table = tabulate(bal.iloc[2:], headers=header_rows, tablefmt="html", showindex=True)
print(bal_table)

# with open(bal_out,"w") as f:
#     f.write(bal_table)

# !cp {bal_out} ../../../../eric-mc2-cv/static/uploads/

<table>
<thead>
<tr><th>                     </th><th style="text-align: right;">          bike
Control</th><th style="text-align: right;">          
Treated</th><th style="text-align: right;">          train
Control</th><th style="text-align: right;">         
Treated</th><th style="text-align: right;">          uber
Control</th><th style="text-align: right;">          
Treated</th></tr>
</thead>
<tbody>
<tr><td>N units              </td><td style="text-align: right;"> 1549    </td><td style="text-align: right;">  47    </td><td style="text-align: right;">  134    </td><td style="text-align: right;">  8    </td><td style="text-align: right;"> 1217    </td><td style="text-align: right;">  40    </td></tr>
<tr><td>N unit-years         </td><td style="text-align: right;">76244    </td><td style="text-align: right;">4552    </td><td style="text-align: right;">13132    </td><td style="text-align: right;">714    </td><td style="text-align: right;">95398    </td><td style="text-align: right;